# Seteo de workspace
Se seteo el path para tener un workspace dinámico

In [0]:
%python
# Obtener el path del notebook actual y construir el path del CSV dinámicamente
import os

# Obtener el path del notebook actual
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# Construir el path del proyecto (subir dos niveles desde /process)
project_root = os.path.dirname(os.path.dirname(notebook_path))

# Path del CSV relativo al proyecto
csv_path = f"/Workspace{project_root}/data-set-challenge-6-.csv"

print(f"Notebook path: {notebook_path}")
print(f"Project root: {project_root}")
print(f"CSV path: {csv_path}")



In [0]:
%python
dbutils.widgets.text("process_datetime", "")
dbutils.widgets.text("runId", "")

process_datetime = dbutils.widgets.get("process_datetime")
runId = dbutils.widgets.get("runId")

In [0]:
%python
import json

In [0]:
%python
df_csv = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(csv_path)
df_csv_filtered = df_csv.filter(f"DATE(fecha) >= '{process_datetime}'")
df_csv_filtered.createOrReplaceTempView("csv_raw_data")

In [0]:
%python
records_read = spark.sql("""
    SELECT count(1) FROM csv_raw_data
""")
if records_read.first()[0] == 0:
    result = {
        "status": "SUCCESS",
        "records_read": 0,
        "records_written": 0,
        "error_message": None,
        "table_name": "bronze.transacciones",
        "layer": "BRONZE"
    }
    dbutils.notebook.exit(json.dumps(result))

In [0]:
%python
try:
    written_records = spark.sql(f"""
        INSERT INTO bronze.transacciones
        SELECT 
            fecha,
            tipoTran,
            id_cliente,
            descripcion_titulo,
            moneda,
            simbolo_titulo,
            cantidad,
            precio,
            id_transaccion,
            origen,
            DATE(fecha) AS fecha_particion,
            TIMESTAMP('{process_datetime}') AS fecha_auditoria
        FROM 
            csv_raw_data
        """
    )
except Exception as e:
    result = {
        "status": "ERROR",
        "records_read": records_read.first()[0],
        "records_written": 0,
        "error_message": str(e),
        "table_name": "bronze.transacciones"
    }
    dbutils.notebook.exit(json.dumps(result))

In [0]:
%python
result = {
    "status": "SUCCESS",
    "records_read": records_read.first()[0],
    "records_written": written_records.first()[0],
    "error_message": None,
    "table_name": "bronze.transacciones",
    "layer": "BRONZE"
}
dbutils.notebook.exit(json.dumps(result))